# GNSS-Denied Cross-View Matching — High-Resolution Training

Train drone↔satellite feature extractor at 0.30 m/px (zoom 18) resolution.
Target: <25m position error on custom satellite reference.

In [ ]:
!pip install -q timm

import os, cv2, glob, random, time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
import requests, math

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/DroneCV/gnss_denied'
DATA_DIR = '/content/data/crossview_hires'
os.makedirs(f'{WORK_DIR}/models', exist_ok=True)
os.makedirs(f'{DATA_DIR}/satellite', exist_ok=True)
os.makedirs(f'{DATA_DIR}/drone', exist_ok=True)
print('Setup complete')


## Step 1: Download High-Res Satellite Tiles (zoom 18, ~0.30 m/px)

We download from Google Maps Static API (no key needed for low volume)
or fall back to direct tile URL patterns.

In [ ]:
def lat_lon_to_tile(lat, lon, zoom):
    n = 2**zoom
    x = int((lon + 180) / 360 * n)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1/math.cos(math.radians(lat))) / math.pi) / 2 * n)
    return x, y

# 500 diverse European locations for training
np.random.seed(42)
cities = [
    (60.17, 24.94, 'Helsinki'), (60.45, 22.27, 'Turku'),
    (61.50, 23.79, 'Tampere'), (59.33, 18.07, 'Stockholm'),
    (55.68, 12.57, 'Copenhagen'), (59.91, 10.75, 'Oslo'),
    (52.52, 13.41, 'Berlin'), (48.86, 2.35, 'Paris'),
    (51.51, -0.13, 'London'), (52.37, 4.90, 'Amsterdam'),
    (50.08, 14.44, 'Prague'), (47.50, 19.04, 'Budapest'),
    (41.39, 2.17, 'Barcelona'), (45.46, 9.19, 'Milan'),
    (48.21, 16.37, 'Vienna'), (53.55, 10.00, 'Hamburg'),
]
locations = []
for lat, lon, name in cities:
    for _ in range(32):
        locations.append((lat + np.random.uniform(-0.03, 0.03),
                          lon + np.random.uniform(-0.05, 0.05)))

print(f'Downloading {len(locations)} tiles at zoom 18...')
session = requests.Session()
session.headers['User-Agent'] = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36'
success = 0

for i, (lat, lon) in enumerate(locations):
    x, y = lat_lon_to_tile(lat, lon, 18)
    # Try multiple tile servers
    urls = [
        f'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z=18',
        f'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/18/{y}/{x}',
        f'https://khms1.google.com/kh/v=984&x={x}&y={y}&z=18',
    ]
    for url in urls:
        try:
            r = session.get(url, timeout=8)
            if r.status_code == 200 and len(r.content) > 5000:
                img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
                if img is not None and img.shape[0] >= 200:
                    cv2.imwrite(f'{DATA_DIR}/satellite/{success:04d}.jpg', img)
                    # Generate augmented drone view
                    h, w = img.shape[:2]
                    crop = int(min(h,w) * np.random.uniform(0.45, 0.85))
                    cx = np.random.randint(0, max(1, w-crop))
                    cy = np.random.randint(0, max(1, h-crop))
                    drone = cv2.resize(img[cy:cy+crop, cx:cx+crop], (256, 256))
                    angle = np.random.uniform(-35, 35)
                    M = cv2.getRotationMatrix2D((128,128), angle, np.random.uniform(0.95, 1.05))
                    drone = cv2.warpAffine(drone, M, (256,256), borderMode=cv2.BORDER_REFLECT)
                    drone = cv2.GaussianBlur(drone, (3,3), 0)
                    drone = np.clip(drone.astype(float) * np.random.uniform(0.75, 1.25) + np.random.uniform(-25, 25), 0, 255).astype(np.uint8)
                    cv2.imwrite(f'{DATA_DIR}/drone/{success:04d}.jpg', drone)
                    success += 1
                    break
        except:
            continue
    if (i+1) % 100 == 0:
        print(f'  {i+1}/{len(locations)} attempted, {success} pairs')
    if success >= 500:
        break

print(f'\nTotal: {success} high-res pairs at ~0.30 m/px')
if success < 50:
    print('WARNING: Tile servers may be rate-limiting. Using EuroSAT fallback...')
    !pip install -q datasets
    from datasets import load_dataset
    ds = load_dataset('blanchon/EuroSAT_RGB', split='train[:2000]')
    for i, sample in enumerate(ds):
        img = cv2.resize(np.array(sample['image']), (256,256))
        cv2.imwrite(f'{DATA_DIR}/satellite/{i:04d}.jpg', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        crop = int(256*np.random.uniform(0.5,0.85))
        x,y = np.random.randint(0,256-crop), np.random.randint(0,256-crop)
        d = cv2.resize(img[y:y+crop,x:x+crop], (256,256))
        M = cv2.getRotationMatrix2D((128,128), np.random.uniform(-25,25), 1.0)
        d = cv2.warpAffine(d, M, (256,256))
        cv2.imwrite(f'{DATA_DIR}/drone/{i:04d}.jpg', cv2.cvtColor(d, cv2.COLOR_RGB2BGR))
    success = len(ds)
    print(f'Fallback: {success} EuroSAT pairs')


In [ ]:
import matplotlib.pyplot as plt
sat_files = sorted(glob.glob(f'{DATA_DIR}/satellite/*.jpg'))
drone_files = sorted(glob.glob(f'{DATA_DIR}/drone/*.jpg'))
print(f'{len(sat_files)} satellite, {len(drone_files)} drone')
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
step = max(1, len(sat_files)//5)
for i in range(5):
    idx = i*step
    axes[0,i].imshow(cv2.cvtColor(cv2.imread(sat_files[idx]), cv2.COLOR_BGR2RGB))
    axes[0,i].set_title(f'Sat {idx}'); axes[0,i].axis('off')
    axes[1,i].imshow(cv2.cvtColor(cv2.imread(drone_files[idx]), cv2.COLOR_BGR2RGB))
    axes[1,i].set_title(f'Drone {idx}'); axes[1,i].axis('off')
plt.suptitle('High-Res Training Pairs (0.30 m/px)'); plt.tight_layout(); plt.show()


## Step 2: Model + Training

In [ ]:
class CrossViewEncoder(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b2', pretrained=True, num_classes=0)
        self.embed = nn.Sequential(
            nn.Linear(self.backbone.num_features, embed_dim),
            nn.BatchNorm1d(embed_dim), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(embed_dim, embed_dim))
    def forward(self, x):
        return nn.functional.normalize(self.embed(self.backbone(x)), p=2, dim=1)

class TripletDataset(Dataset):
    def __init__(self, data_dir, augment=True):
        self.sat = sorted(glob.glob(f'{data_dir}/satellite/*.jpg'))
        self.drone = sorted(glob.glob(f'{data_dir}/drone/*.jpg'))
        self.n = min(len(self.sat), len(self.drone))
        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((224,224)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(15),
                transforms.ColorJitter(0.3, 0.3, 0.2, 0.05),
                transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224,224)), transforms.ToTensor(),
                transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    def __len__(self): return self.n * 3  # 3 passes per epoch
    def __getitem__(self, idx):
        i = idx % self.n
        drone = self.transform(Image.open(self.drone[i]).convert('RGB'))
        sat_pos = self.transform(Image.open(self.sat[i]).convert('RGB'))
        # Hard negative: nearby index (geographically close = harder)
        offset = random.choice(range(max(1, self.n//20), self.n-1))
        neg_idx = (i + offset) % self.n
        sat_neg = self.transform(Image.open(self.sat[neg_idx]).convert('RGB'))
        return drone, sat_pos, sat_neg

model = CrossViewEncoder().cuda()
dataset = TripletDataset(DATA_DIR)
loader = DataLoader(dataset, batch_size=48, shuffle=True, num_workers=4, pin_memory=True)
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params (EfficientNet-B2)')
print(f'Dataset: {dataset.n} pairs, {len(loader)} batches/epoch')


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
criterion = nn.TripletMarginLoss(margin=0.5)

EPOCHS = 30
t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    losses = []
    for drone, sat_pos, sat_neg in loader:
        drone, sat_pos, sat_neg = drone.cuda(), sat_pos.cuda(), sat_neg.cuda()
        loss = criterion(model(drone), model(sat_pos), model(sat_neg))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} — Loss: {np.mean(losses):.4f} ({time.time()-t0:.0f}s)')

torch.save(model.state_dict(), f'{WORK_DIR}/models/crossview_effb2_512d_v2.pth')
print(f'\nTraining complete ({time.time()-t0:.0f}s). Model saved.')


## Step 3: Evaluate + Test on Jorvas

In [ ]:
model.eval()
eval_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# Retrieval accuracy on training data (sanity check)
sat_files = sorted(glob.glob(f'{DATA_DIR}/satellite/*.jpg'))
drone_files = sorted(glob.glob(f'{DATA_DIR}/drone/*.jpg'))
n_eval = min(200, len(sat_files))

def embed_files(files):
    embs = []
    for i in range(0, len(files), 64):
        batch = torch.stack([eval_tf(Image.open(f).convert('RGB')) for f in files[i:i+64]]).cuda()
        with torch.no_grad(): embs.append(model(batch).cpu().numpy())
    return np.vstack(embs)

sat_emb = embed_files(sat_files[:n_eval])
drone_emb = embed_files(drone_files[:n_eval])
sims = drone_emb @ sat_emb.T
top1 = sum(np.argmax(sims[i]) == i for i in range(n_eval))
top5 = sum(i in np.argsort(-sims[i])[:5] for i in range(n_eval))
print(f'Retrieval (n={n_eval}): Top-1={top1/n_eval*100:.1f}%, Top-5={top5/n_eval*100:.1f}%')


In [ ]:
# Test on custom satellite reference image
from google.colab import files
print('Upload satellite reference (e.g., jorvas_satellite_z18_stitched.jpg):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    ref = cv2.imread(fname)
    h, w = ref.shape[:2]
    print(f'Reference: {ref.shape} ({h*0.30:.0f}m × {w*0.30:.0f}m at 0.30 m/px)')

    # Dense tiling with 64px stride → ~19m per tile center
    TILE = 224
    STRIDE = 48  # ~14m stride for <25m target
    tiles_emb, centers = [], []
    for row in range(0, h-TILE+1, STRIDE):
        for col in range(0, w-TILE+1, STRIDE):
            tile = ref[row:row+TILE, col:col+TILE]
            t = eval_tf(Image.fromarray(cv2.cvtColor(tile, cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
            with torch.no_grad(): tiles_emb.append(model(t).cpu().numpy().flatten())
            centers.append((col + TILE//2, row + TILE//2))
    tiles_emb = np.array(tiles_emb)
    print(f'Embedded {len(tiles_emb)} tiles (stride={STRIDE}px={STRIDE*0.30:.1f}m)')

    # Test multiple simulated drone positions
    errors = []
    for test_idx in range(10):
        # Random position in the reference
        tx = np.random.randint(TILE, w-TILE)
        ty = np.random.randint(TILE, h-TILE)
        # Crop drone view with augmentation
        crop_sz = int(TILE * np.random.uniform(0.7, 1.0))
        cx = tx - crop_sz//2
        cy = ty - crop_sz//2
        drone_crop = ref[max(0,cy):cy+crop_sz, max(0,cx):cx+crop_sz]
        drone_crop = cv2.resize(drone_crop, (224, 224))
        angle = np.random.uniform(-20, 20)
        M = cv2.getRotationMatrix2D((112,112), angle, 1.0)
        drone_crop = cv2.warpAffine(drone_crop, M, (224,224), borderMode=cv2.BORDER_REFLECT)
        
        d = eval_tf(Image.fromarray(cv2.cvtColor(drone_crop, cv2.COLOR_BGR2RGB))).unsqueeze(0).cuda()
        with torch.no_grad(): d_emb = model(d).cpu().numpy().flatten()
        
        sims = tiles_emb @ d_emb
        best = np.argmax(sims)
        est = centers[best]
        err_px = np.sqrt((est[0]-tx)**2 + (est[1]-ty)**2)
        err_m = err_px * 0.30
        errors.append(err_m)
        print(f'  Test {test_idx+1}: true=({tx},{ty}), est={est}, error={err_m:.1f}m')

    print(f'\n=== RESULTS ===')
    print(f'Mean error: {np.mean(errors):.1f}m')
    print(f'Median error: {np.median(errors):.1f}m')
    print(f'Best: {np.min(errors):.1f}m, Worst: {np.max(errors):.1f}m')
    print(f'Under 25m: {sum(1 for e in errors if e < 25)}/{len(errors)}')
